## Exercise Sheet No. 1
### Exercise 3

#### Phase 1: Reading and parsing the file in python


Importing relevant packages

In [56]:
import pandas as pd
import pathlib
from pathlib import Path

import duckdb

In [18]:
presidents_data_path = Path.cwd() / "data" / "presidents.txt"
presidents_data_path

PosixPath('/Users/AnanyaPal1/Documents/data-engineering/notebooks/information_systems/data/presidents.txt')

In [19]:
with open(presidents_data_path, "r") as f:
    content = f.read()

content

'#presidents:[(Washington:Federalist:Virginia);(Jefferson:DemoRep:Virginia);(Harrison:Whig:Virginia);(Lincoln:Republican:Kentucky);(Cleveland:Democratic:NewJersey);(Roosevelt:Republican:NewYork);(Kennedy:Democratic:Massachusetts);(Clinton:Democratic:Arkansas);(Bush:Republican:Texas);(Obama:Democratic:Hawaii);(Trump:Republican:NewYork);(Biden:Democratic:Pennsylvania)]\n#hobby:[(Jefferson:Fishing);(Jefferson:Riding);(Harrison:Hunting);(Lincoln:Walking);(Cleveland:Fishing);(Roosevelt:Boxing);(Roosevelt:Hunting);(Roosevelt:Jujitsu);(Roosevelt:Riding);(Roosevelt:Shooting);(Roosevelt:Tennis);(Roosevelt:Wrestling);(Kennedy:Sailing);(Kennedy:Swimming);(Kennedy:TouchFootball);(Clinton:Saxophone);(Bush:Baseball);(Bush:Golf);(Bush:Fishing);(Obama:Basketball);(Obama:Poker);(Obama:Cooking);(Trump:Golf);(Trump:WatchingTV);(Trump:PublicSpeaking);(Biden:Exercise);(Biden:Sports);(Biden:Reading)]\n'

In [20]:
parts = content.split("#hobby:")
presidents_part = parts[0]
hobbies_part = parts[1]

display(presidents_part)
display(hobbies_part)

'#presidents:[(Washington:Federalist:Virginia);(Jefferson:DemoRep:Virginia);(Harrison:Whig:Virginia);(Lincoln:Republican:Kentucky);(Cleveland:Democratic:NewJersey);(Roosevelt:Republican:NewYork);(Kennedy:Democratic:Massachusetts);(Clinton:Democratic:Arkansas);(Bush:Republican:Texas);(Obama:Democratic:Hawaii);(Trump:Republican:NewYork);(Biden:Democratic:Pennsylvania)]\n'

'[(Jefferson:Fishing);(Jefferson:Riding);(Harrison:Hunting);(Lincoln:Walking);(Cleveland:Fishing);(Roosevelt:Boxing);(Roosevelt:Hunting);(Roosevelt:Jujitsu);(Roosevelt:Riding);(Roosevelt:Shooting);(Roosevelt:Tennis);(Roosevelt:Wrestling);(Kennedy:Sailing);(Kennedy:Swimming);(Kennedy:TouchFootball);(Clinton:Saxophone);(Bush:Baseball);(Bush:Golf);(Bush:Fishing);(Obama:Basketball);(Obama:Poker);(Obama:Cooking);(Trump:Golf);(Trump:WatchingTV);(Trump:PublicSpeaking);(Biden:Exercise);(Biden:Sports);(Biden:Reading)]\n'

In [21]:
# remove the labels
presidents_part = presidents_part.replace("#presidents:", "").strip()
hobbies_part = hobbies_part.strip()

display(presidents_part)
display(hobbies_part)

'[(Washington:Federalist:Virginia);(Jefferson:DemoRep:Virginia);(Harrison:Whig:Virginia);(Lincoln:Republican:Kentucky);(Cleveland:Democratic:NewJersey);(Roosevelt:Republican:NewYork);(Kennedy:Democratic:Massachusetts);(Clinton:Democratic:Arkansas);(Bush:Republican:Texas);(Obama:Democratic:Hawaii);(Trump:Republican:NewYork);(Biden:Democratic:Pennsylvania)]'

'[(Jefferson:Fishing);(Jefferson:Riding);(Harrison:Hunting);(Lincoln:Walking);(Cleveland:Fishing);(Roosevelt:Boxing);(Roosevelt:Hunting);(Roosevelt:Jujitsu);(Roosevelt:Riding);(Roosevelt:Shooting);(Roosevelt:Tennis);(Roosevelt:Wrestling);(Kennedy:Sailing);(Kennedy:Swimming);(Kennedy:TouchFootball);(Clinton:Saxophone);(Bush:Baseball);(Bush:Golf);(Bush:Fishing);(Obama:Basketball);(Obama:Poker);(Obama:Cooking);(Trump:Golf);(Trump:WatchingTV);(Trump:PublicSpeaking);(Biden:Exercise);(Biden:Sports);(Biden:Reading)]'

In [27]:
entries = presidents_part.strip("[]").split(";")
entries

['(Washington:Federalist:Virginia)',
 '(Jefferson:DemoRep:Virginia)',
 '(Harrison:Whig:Virginia)',
 '(Lincoln:Republican:Kentucky)',
 '(Cleveland:Democratic:NewJersey)',
 '(Roosevelt:Republican:NewYork)',
 '(Kennedy:Democratic:Massachusetts)',
 '(Clinton:Democratic:Arkansas)',
 '(Bush:Republican:Texas)',
 '(Obama:Democratic:Hawaii)',
 '(Trump:Republican:NewYork)',
 '(Biden:Democratic:Pennsylvania)']

In [30]:
cleaned_entries = []

for entry in entries:
    cleaned = entry.strip("()")
    cleaned_entries.append(cleaned)

cleaned_entries

['Washington:Federalist:Virginia',
 'Jefferson:DemoRep:Virginia',
 'Harrison:Whig:Virginia',
 'Lincoln:Republican:Kentucky',
 'Cleveland:Democratic:NewJersey',
 'Roosevelt:Republican:NewYork',
 'Kennedy:Democratic:Massachusetts',
 'Clinton:Democratic:Arkansas',
 'Bush:Republican:Texas',
 'Obama:Democratic:Hawaii',
 'Trump:Republican:NewYork',
 'Biden:Democratic:Pennsylvania']

In [36]:
final_presidents = []
for parts in cleaned_entries:
    parts = parts.split(":")
    final_presidents.append(parts)

final_presidents

[['Washington', 'Federalist', 'Virginia'],
 ['Jefferson', 'DemoRep', 'Virginia'],
 ['Harrison', 'Whig', 'Virginia'],
 ['Lincoln', 'Republican', 'Kentucky'],
 ['Cleveland', 'Democratic', 'NewJersey'],
 ['Roosevelt', 'Republican', 'NewYork'],
 ['Kennedy', 'Democratic', 'Massachusetts'],
 ['Clinton', 'Democratic', 'Arkansas'],
 ['Bush', 'Republican', 'Texas'],
 ['Obama', 'Democratic', 'Hawaii'],
 ['Trump', 'Republican', 'NewYork'],
 ['Biden', 'Democratic', 'Pennsylvania']]

In [40]:
hobbies_clean = hobbies_part.strip("[]")

entries = hobbies_clean.split(";")

entries = [e.strip("()") for e in entries]

final_hobbies = [e.split(":") for e in entries]

final_hobbies

[['Jefferson', 'Fishing'],
 ['Jefferson', 'Riding'],
 ['Harrison', 'Hunting'],
 ['Lincoln', 'Walking'],
 ['Cleveland', 'Fishing'],
 ['Roosevelt', 'Boxing'],
 ['Roosevelt', 'Hunting'],
 ['Roosevelt', 'Jujitsu'],
 ['Roosevelt', 'Riding'],
 ['Roosevelt', 'Shooting'],
 ['Roosevelt', 'Tennis'],
 ['Roosevelt', 'Wrestling'],
 ['Kennedy', 'Sailing'],
 ['Kennedy', 'Swimming'],
 ['Kennedy', 'TouchFootball'],
 ['Clinton', 'Saxophone'],
 ['Bush', 'Baseball'],
 ['Bush', 'Golf'],
 ['Bush', 'Fishing'],
 ['Obama', 'Basketball'],
 ['Obama', 'Poker'],
 ['Obama', 'Cooking'],
 ['Trump', 'Golf'],
 ['Trump', 'WatchingTV'],
 ['Trump', 'PublicSpeaking'],
 ['Biden', 'Exercise'],
 ['Biden', 'Sports'],
 ['Biden', 'Reading']]

Converting into a dictionary format to feed into the database

In [42]:
presidents_dict = [
    {
        "name": p[0],
        "party": p[1],
        "state": p[2]
    }
    for p in final_presidents
]

presidents_dict

[{'name': 'Washington', 'party': 'Federalist', 'state': 'Virginia'},
 {'name': 'Jefferson', 'party': 'DemoRep', 'state': 'Virginia'},
 {'name': 'Harrison', 'party': 'Whig', 'state': 'Virginia'},
 {'name': 'Lincoln', 'party': 'Republican', 'state': 'Kentucky'},
 {'name': 'Cleveland', 'party': 'Democratic', 'state': 'NewJersey'},
 {'name': 'Roosevelt', 'party': 'Republican', 'state': 'NewYork'},
 {'name': 'Kennedy', 'party': 'Democratic', 'state': 'Massachusetts'},
 {'name': 'Clinton', 'party': 'Democratic', 'state': 'Arkansas'},
 {'name': 'Bush', 'party': 'Republican', 'state': 'Texas'},
 {'name': 'Obama', 'party': 'Democratic', 'state': 'Hawaii'},
 {'name': 'Trump', 'party': 'Republican', 'state': 'NewYork'},
 {'name': 'Biden', 'party': 'Democratic', 'state': 'Pennsylvania'}]

In [43]:
hobbies_dict = [
    {
        "name": p[0],
        "hobby": p[1]
    }
    for p in final_hobbies
]

hobbies_dict

[{'name': 'Jefferson', 'hobby': 'Fishing'},
 {'name': 'Jefferson', 'hobby': 'Riding'},
 {'name': 'Harrison', 'hobby': 'Hunting'},
 {'name': 'Lincoln', 'hobby': 'Walking'},
 {'name': 'Cleveland', 'hobby': 'Fishing'},
 {'name': 'Roosevelt', 'hobby': 'Boxing'},
 {'name': 'Roosevelt', 'hobby': 'Hunting'},
 {'name': 'Roosevelt', 'hobby': 'Jujitsu'},
 {'name': 'Roosevelt', 'hobby': 'Riding'},
 {'name': 'Roosevelt', 'hobby': 'Shooting'},
 {'name': 'Roosevelt', 'hobby': 'Tennis'},
 {'name': 'Roosevelt', 'hobby': 'Wrestling'},
 {'name': 'Kennedy', 'hobby': 'Sailing'},
 {'name': 'Kennedy', 'hobby': 'Swimming'},
 {'name': 'Kennedy', 'hobby': 'TouchFootball'},
 {'name': 'Clinton', 'hobby': 'Saxophone'},
 {'name': 'Bush', 'hobby': 'Baseball'},
 {'name': 'Bush', 'hobby': 'Golf'},
 {'name': 'Bush', 'hobby': 'Fishing'},
 {'name': 'Obama', 'hobby': 'Basketball'},
 {'name': 'Obama', 'hobby': 'Poker'},
 {'name': 'Obama', 'hobby': 'Cooking'},
 {'name': 'Trump', 'hobby': 'Golf'},
 {'name': 'Trump', 'hobby'

Checking dictionary parsing

In [44]:
presidents_dict[0]

{'name': 'Washington', 'party': 'Federalist', 'state': 'Virginia'}

In [45]:
hobbies_dict[0]

{'name': 'Jefferson', 'hobby': 'Fishing'}

In [46]:
len(presidents_dict)

12

In [47]:
len(hobbies_dict)

28

In [51]:
presidents_name = [p["name"] for p in presidents_dict]
len(presidents_name)

12

In [53]:
hobbys_name = [p["name"] for p in hobbies_dict]
len(hobbys_name)

28

In [54]:
unique_presidents = set(presidents_name)
unique_hobby_names = set(hobbys_name)

len(unique_presidents)
len(unique_hobby_names)

11

In [55]:
unique_hobby_names.issubset(unique_presidents)

True

### Phase 2: Importing to DuckDB

In [57]:
con = duckdb.connect("presidents.db")

In [60]:
con.execute("""
CREATE TABLE presidents(
    name VARCHAR,
    party VARCHAR,
    state VARCHAR,           
)
""")

con.execute("""
CREATE TABLE hobbies(
    name VARCHAR,
    hobby VARCHAR           
)
""")

# during testing
con.execute("DROP TABLE IF EXISTS presidents")
con.execute("DROP TABLE IF EXISTS hobbies")

CatalogException: Catalog Error: Table with name "presidents" already exists!

### Load Python data

In [61]:
df_presidents = pd.DataFrame(presidents_dict)
df_hobbies = pd.DataFrame(hobbies_dict)

In [62]:
con.register("df_presidents", df_presidents)
con.register("df_hobbies", df_hobbies)

In [67]:
con.execute("DROP TABLE IF EXISTS presidents")
con.execute("DROP TABLE IF EXISTS hobbies")

In [68]:
con.execute("CREATE TABLE presidents " \
"AS SELECT *" \
"FROM df_presidents")

con.execute("CREATE TABLE hobbies " \
"AS SELECT *" \
"FROM df_hobbies")

In [69]:
con.execute("SELECT * FROM presidents").fetchall()

[('Washington', 'Federalist', 'Virginia'),
 ('Jefferson', 'DemoRep', 'Virginia'),
 ('Harrison', 'Whig', 'Virginia'),
 ('Lincoln', 'Republican', 'Kentucky'),
 ('Cleveland', 'Democratic', 'NewJersey'),
 ('Roosevelt', 'Republican', 'NewYork'),
 ('Kennedy', 'Democratic', 'Massachusetts'),
 ('Clinton', 'Democratic', 'Arkansas'),
 ('Bush', 'Republican', 'Texas'),
 ('Obama', 'Democratic', 'Hawaii'),
 ('Trump', 'Republican', 'NewYork'),
 ('Biden', 'Democratic', 'Pennsylvania')]

In [70]:
con.execute("SELECT * FROM hobbies").fetchall()

[('Jefferson', 'Fishing'),
 ('Jefferson', 'Riding'),
 ('Harrison', 'Hunting'),
 ('Lincoln', 'Walking'),
 ('Cleveland', 'Fishing'),
 ('Roosevelt', 'Boxing'),
 ('Roosevelt', 'Hunting'),
 ('Roosevelt', 'Jujitsu'),
 ('Roosevelt', 'Riding'),
 ('Roosevelt', 'Shooting'),
 ('Roosevelt', 'Tennis'),
 ('Roosevelt', 'Wrestling'),
 ('Kennedy', 'Sailing'),
 ('Kennedy', 'Swimming'),
 ('Kennedy', 'TouchFootball'),
 ('Clinton', 'Saxophone'),
 ('Bush', 'Baseball'),
 ('Bush', 'Golf'),
 ('Bush', 'Fishing'),
 ('Obama', 'Basketball'),
 ('Obama', 'Poker'),
 ('Obama', 'Cooking'),
 ('Trump', 'Golf'),
 ('Trump', 'WatchingTV'),
 ('Trump', 'PublicSpeaking'),
 ('Biden', 'Exercise'),
 ('Biden', 'Sports'),
 ('Biden', 'Reading')]

In [72]:
display(con.execute("SELECT COUNT(*) FROM presidents").fetchone())
display(con.execute("SELECT COUNT(*) FROM hobbies").fetchone())

(12,)

(28,)

## Build a relationship schema

In [73]:
con.execute("""
SELECT p.name, h.hobby
FROM presidents p
JOIN hobbies h
ON p.name = h.name
LIMIT 10
""").fetchall()

[('Jefferson', 'Fishing'),
 ('Jefferson', 'Riding'),
 ('Harrison', 'Hunting'),
 ('Lincoln', 'Walking'),
 ('Cleveland', 'Fishing'),
 ('Roosevelt', 'Boxing'),
 ('Roosevelt', 'Hunting'),
 ('Roosevelt', 'Jujitsu'),
 ('Roosevelt', 'Riding'),
 ('Roosevelt', 'Shooting')]

2a- print names of all presidents who belong or belonged to the Republican party

In [ ]:
[row [0] for row in con.execute("""
SELECT name FROM presidents WHERE party = 'Republican'
""").fetchall()]

['Lincoln', 'Roosevelt', 'Bush', 'Trump']

2b- print the names of all presidents who belong or belonged to the Republican party in alphabetical order.

In [84]:
[row[0] for row in con.execute("""
SELECT name FROM presidents WHERE party = 'Republican' ORDER BY name ASC
""").fetchall()]

['Bush', 'Lincoln', 'Roosevelt', 'Trump']

2c- the hobbies of all presidents who do not belong to the Democratic party. These should be printed as pairs of the form (party:hobby)

In [89]:
con.execute("""
SELECT p.party || ':' ||h.hobby
FROM presidents p
JOIN hobbies h
ON p.name = h.name
WHERE p.party != 'Democratic'
""").fetchall()

[('DemoRep:Fishing',),
 ('DemoRep:Riding',),
 ('Whig:Hunting',),
 ('Republican:Walking',),
 ('Republican:Boxing',),
 ('Republican:Hunting',),
 ('Republican:Jujitsu',),
 ('Republican:Riding',),
 ('Republican:Shooting',),
 ('Republican:Tennis',),
 ('Republican:Wrestling',),
 ('Republican:Baseball',),
 ('Republican:Golf',),
 ('Republican:Fishing',),
 ('Republican:Golf',),
 ('Republican:WatchingTV',),
 ('Republican:PublicSpeaking',)]

2d- for each president, the number of his hobbies, printed as pairs of the form (name:number_of_hobbies).

In [92]:
[row[0] for row in con.execute("""
SELECT p.name || ':' || COUNT(h.hobby)
FROM presidents p
JOIN hobbies h
ON p.name = h.name
GROUP BY p.name
""").fetchall()]

['Harrison:1',
 'Kennedy:3',
 'Clinton:1',
 'Lincoln:1',
 'Jefferson:2',
 'Cleveland:1',
 'Roosevelt:7',
 'Trump:3',
 'Obama:3',
 'Bush:3',
 'Biden:3']